In [1]:
# ---------------------------------------
# IMPORTS
# ---------------------------------------
import time
import csv
from typing import Dict, Any, List
from datetime import datetime, timedelta, timezone
import requests
import pandas as pd 

/Users/usuario/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv(Path("..") / ".env")
ACCESS_TOKEN = os.getenv('ACCESS_TOKEN')
IG_USER_ID = os.getenv('IG_USER_ID')

In [3]:
from dotenv import load_dotenv, find_dotenv
import os

dotenv_path = find_dotenv()  # busca .env hacia arriba
print("Usando .env en:", dotenv_path)

load_dotenv(dotenv_path)

ACCESS_TOKEN = os.getenv("ACCESS_TOKEN")
IG_USER_ID = os.getenv("IG_USER_ID")

print("ACCESS_TOKEN es None?:", ACCESS_TOKEN is None)
print("IG_USER_ID es None?:", IG_USER_ID is None)

Usando .env en: /Users/usuario/Desktop/ADALAB/Proyectos/enGira!/EXTRACCION API IG/.env
ACCESS_TOKEN es None?: False
IG_USER_ID es None?: False


In [4]:
print("ACCESS_TOKEN:", ACCESS_TOKEN)
print("IG_USER_ID:", IG_USER_ID)

ACCESS_TOKEN: EAAXhnqhX1nsBQ1aZCquoTSSGbZBhdbQhnp9y1ZA4AI6xFsSDlptdD1ZBmjotI7jI3KrUH8RqeZCpCCDKTMoAawaO6dNs4LsSZAaAY9W61kpq9a2EsPbwGxtUsmk81qHje5OAJvNKGDXH2RMP8hTZCQ21pqt7rt2kJ4mstZA5UwwtyoZBzYcsTBEPyuk9B97ZB2ZCzwFb9rgulmCcsI9SWGrF8E8OLVXZBkoC6HwMt0wbFQkgCraOhIgmwfGB1c8saoBOcQBSO8aWlYpEW7tBqF5LPwVa7QZDZD
IG_USER_ID: 17841467002220104


In [5]:
IG_MEDIA_ID = "18071586206348174"

url = f"https://graph.facebook.com/v21.0/{IG_MEDIA_ID}/insights"
params = {
    "metric": "views,reach",
    "access_token": ACCESS_TOKEN
}

r = requests.get(url, params=params)
print(r.status_code)
print(r.json())

200
{'data': [{'name': 'views', 'period': 'lifetime', 'values': [{'value': 1724}], 'title': 'Reproducciones', 'description': 'Número de veces que se reprodujeron o se mostraron tus reels', 'id': '18071586206348174/insights/views/lifetime'}, {'name': 'reach', 'period': 'lifetime', 'values': [{'value': 1031}], 'title': 'Cuentas alcanzadas', 'description': 'Número de cuentas únicas que vieron este reel al menos una vez. El alcance es diferente de las impresiones, que pueden incluir varias reproducciones de los reels por parte de las mismas cuentas. Esta métrica es una estimación.', 'id': '18071586206348174/insights/reach/lifetime'}]}


In [6]:

API_VERSION = "v21.0"
BASE_URL = f"https://graph.facebook.com/{API_VERSION}"

def get_media_list(ig_user_id, access_token, limit=25):
    url = f"{BASE_URL}/{ig_user_id}/media"
    params = {
        "fields": "id,media_type,caption,permalink,timestamp",
        "limit": limit,
        "access_token": access_token
    }
    r = requests.get(url, params=params)
    r.raise_for_status()
    return r.json()

def get_media_insights(media_id, media_type, access_token):
    """
    Devuelve insights para un media según su tipo.
    Si alguna métrica no es soportada, la API devolverá error y te diré cuál ajustar.
    """
    # Sets iniciales razonables (los afinamos con errores reales)
    if media_type == "VIDEO":
        metrics = "views,reach"   # en tu caso ya sabemos que funciona para reels/video
    elif media_type == "IMAGE":
        metrics = "reach"         # conservador; luego añadimos si procede
    elif media_type == "CAROUSEL_ALBUM":
        metrics = "reach"
    else:
        metrics = "reach"

    url = f"{BASE_URL}/{media_id}/insights"
    params = {"metric": metrics, "access_token": access_token}
    r = requests.get(url, params=params)
    return r.status_code, r.json()

# Ejemplo: coger 10 medias y pedir insights
media_resp = get_media_list(IG_USER_ID, ACCESS_TOKEN, limit=10)
media_items = media_resp.get("data", [])

results = []
for item in media_items:
    media_id = item["id"]
    media_type = item.get("media_type")
    status, data = get_media_insights(media_id, media_type, ACCESS_TOKEN)

    results.append({
        "media_id": media_id,
        "media_type": media_type,
        "permalink": item.get("permalink"),
        "timestamp": item.get("timestamp"),
        "status": status,
        "raw": data
    })

    time.sleep(0.2)  # suaviza rate limits

results[:2]

[{'media_id': '18071586206348174',
  'media_type': 'VIDEO',
  'permalink': 'https://www.instagram.com/reel/DRZFycCDeCe/',
  'timestamp': '2025-11-23T08:29:17+0000',
  'status': 200,
  'raw': {'data': [{'name': 'views',
     'period': 'lifetime',
     'values': [{'value': 1724}],
     'title': 'Reproducciones',
     'description': 'Número de veces que se reprodujeron o se mostraron tus reels',
     'id': '18071586206348174/insights/views/lifetime'},
    {'name': 'reach',
     'period': 'lifetime',
     'values': [{'value': 1031}],
     'title': 'Cuentas alcanzadas',
     'description': 'Número de cuentas únicas que vieron este reel al menos una vez. El alcance es diferente de las impresiones, que pueden incluir varias reproducciones de los reels por parte de las mismas cuentas. Esta métrica es una estimación.',
     'id': '18071586206348174/insights/reach/lifetime'}]}},
 {'media_id': '17924873466153742',
  'media_type': 'VIDEO',
  'permalink': 'https://www.instagram.com/reel/DP30BFgDXFU

In [7]:

API_VERSION = "v21.0"
BASE_URL = f"https://graph.facebook.com/{API_VERSION}"

# 1) Métricas por tipo (Pack C)
REELS_METRICS = [
    "views", "reach", "likes", "comments", "saved", "shares", "total_interactions",
    "ig_reels_avg_watch_time", "ig_reels_video_view_total_time"
]

FEED_METRICS = [
    "reach", "views", "likes", "comments", "saved", "shares", "total_interactions",
    "profile_activity", "profile_visits"
]

def get_media_list_all(ig_user_id, access_token, page_limit=50, max_pages=50, sleep_s=0.2):
    url = f"{BASE_URL}/{ig_user_id}/media"
    params = {
        "fields": "id,media_type,caption,permalink,timestamp",
        "limit": page_limit,
        "access_token": access_token
    }

    all_items = []
    pages = 0

    while url and pages < max_pages:
        r = requests.get(url, params=params)

        if r.status_code != 200:
            # imprime el motivo real y corta
            try:
                err = r.json()
            except Exception:
                err = {"raw_text": r.text}
            raise RuntimeError(f"Graph API error {r.status_code}: {err}")

        js = r.json()
        all_items.extend(js.get("data", []))

        url = js.get("paging", {}).get("next")
        params = None
        pages += 1
        time.sleep(sleep_s)

    return all_items

def insights_request(media_id, metrics, access_token, extra_params=None):
    """
    Hace la petición de insights para un media_id con una lista de metrics.
    extra_params permite cosas como breakdown o period si aplica.
    """
    url = f"{BASE_URL}/{media_id}/insights"
    params = {"metric": ",".join(metrics), "access_token": access_token}
    if extra_params:
        params.update(extra_params)

    r = requests.get(url, params=params)
    return r.status_code, r.json()

def flatten_insights(raw):
    """
    Convierte el JSON de insights a dict plano: {"views": 123, "reach": 45, ...}
    """
    out = {}
    for m in raw.get("data", []):
        name = m.get("name")
        values = m.get("values", [])
        out[name] = values[0].get("value") if values else None
    return out

def get_insights_safe(media_id, media_type, access_token, mode="feed"):
    """
    Pide insights con tolerancia a fallos:
    - Si la API rechaza alguna métrica, la elimina y reintenta.
    mode:
      - "reels": fuerza set REELS
      - "feed": fuerza set FEED
      - "auto": usa media_type para decidir
    """
    if mode == "reels":
        metrics = REELS_METRICS.copy()
    elif mode == "feed":
        metrics = FEED_METRICS.copy()
    else:
        # auto
        metrics = REELS_METRICS.copy() if media_type == "VIDEO" else FEED_METRICS.copy()

    dropped = []
    while metrics:
        status, js = insights_request(media_id, metrics, access_token)
        if status == 200:
            return {
                "status": 200,
                "metrics_used": metrics,
                "metrics_dropped": dropped,
                "raw": js
            }

        # Si falla, intentamos identificar qué métrica es la culpable.
        # Meta suele devolver mensajes del estilo: "... does not support the X metric ..."
        msg = (js.get("error", {}) or {}).get("message", "")
        culprit = None
        for m in metrics:
            if f" {m} " in f" {msg} " or f"'{m}'" in msg or f" {m} metric" in msg:
                culprit = m
                break

        # Si no podemos detectar cuál es, quitamos la última como fallback (para no bloquearnos)
        if culprit is None:
            culprit = metrics[-1]

        metrics.remove(culprit)
        dropped.append(culprit)

        time.sleep(0.1)

    return {
        "status": status,
        "metrics_used": [],
        "metrics_dropped": dropped,
        "raw": js
    }

def build_insights_df(media_items, access_token, sleep_s=0.25):
    """
    - Detecta reels vs feed por permalink (si contiene /reel/)
    - Pide insights con el set Pack C adecuado
    - Devuelve DataFrame plano listo para análisis/CSV
    """
    rows = []

    for item in media_items:
        media_id = item["id"]
        media_type = item.get("media_type")
        permalink = item.get("permalink", "") or ""

        is_reel = "/reel/" in permalink  # truco práctico: reel permalink
        mode = "reels" if is_reel else "feed"

        res = get_insights_safe(media_id, media_type, access_token, mode=mode)
        metrics_flat = flatten_insights(res.get("raw", {}))

        row = {
            "media_id": media_id,
            "media_type": media_type,
            "is_reel": is_reel,
            "permalink": permalink,
            "timestamp": item.get("timestamp"),
            "caption": item.get("caption"),
            "status": res["status"],
            "metrics_used": ",".join(res["metrics_used"]),
            "metrics_dropped": ",".join(res["metrics_dropped"]),
        }
        row.update(metrics_flat)
        rows.append(row)

        time.sleep(sleep_s)

    return pd.DataFrame(rows)

# ===== USO =====
# 1) Traer todos los medias (o pon max_pages más bajo si quieres probar)
media_items = get_media_list_all(IG_USER_ID, ACCESS_TOKEN, page_limit=50, max_pages=50)

# 2) Construir el dataframe con insights
df_insights = build_insights_df(media_items, ACCESS_TOKEN)

df_insights.head()

,media_id,media_type,is_reel,permalink,timestamp,caption,status,metrics_used,metrics_dropped,views,reach,likes,comments,saved,shares,total_interactions,ig_reels_avg_watch_time,ig_reels_video_view_total_time,profile_activity,profile_visits
0,18071586206348174,VIDEO,True,https://www.instagram.com/reel/DRZFycCDeCe/,2025-11-23T08:29:17+0000,🗓️ 𝗣𝗿𝗼́𝘅𝗶𝗺𝗼 𝗺𝗶𝗲́𝗿𝗰𝗼𝗹𝗲𝘀 𝟮𝟲 𝗱𝗲 𝗻𝗼𝘃𝗶𝗲𝗺𝗯𝗿𝗲 \n🕢 A l...,200,"views,reach,likes,comments,saved,shares,total_...",,1724,1031,39,10,15,12,77,8178.0,9323287.0,NaN,NaN
1,17924873466153742,VIDEO,True,https://www.instagram.com/reel/DP30BFgDXFU/,2025-10-16T13:49:33+0000,Acércate a nuestro stand 241 en @mercartes_es ...,200,"views,reach,likes,comments,saved,shares,total_...",,1844,1086,42,2,2,13,59,5467.0,7101837.0,NaN,NaN
2,18134337274395559,IMAGE,False,https://www.instagram.com/p/DHZpx0Bt3I9/,2025-03-20T00:20:32+0000,Pasé por dFeria. Fui con ganas de hablar de la...,200,"reach,views,likes,comments,saved,shares,total_...",,1011,213,22,1,0,0,23,NaN,NaN,0.0,6.0
3,18052603295141608,IMAGE,False,https://www.instagram.com/p/DG5huktNbjs/,2025-03-07T12:54:29+0000,📢¡Atención programadorxs!\n\nCon la 𝗯ú𝘀𝗾𝘂𝗲𝗱𝗮 𝗮...,200,"reach,views,likes,comments,saved,shares,total_...",,1014,247,17,0,6,0,23,NaN,NaN,4.0,7.0
4,18087265219605722,CAROUSEL_ALBUM,False,https://www.instagram.com/p/DGfTzm8th8l/,2025-02-25T08:32:35+0000,"📢¡Atención programadores!\n\n𝟭, 𝟮, 𝟯... ¡𝗽𝗲𝗿𝗳𝗼...",200,"reach,views,likes,comments,saved,shares,total_...",,865,132,3,0,0,0,3,NaN,NaN,0.0,1.0


In [8]:
df_insights.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 20 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   media_id                        24 non-null     object 
 1   media_type                      24 non-null     object 
 2   is_reel                         24 non-null     bool   
 3   permalink                       24 non-null     object 
 4   timestamp                       24 non-null     object 
 5   caption                         21 non-null     object 
 6   status                          24 non-null     int64  
 7   metrics_used                    24 non-null     object 
 8   metrics_dropped                 24 non-null     object 
 9   views                           24 non-null     int64  
 10  reach                           24 non-null     int64  
 11  likes                           24 non-null     int64  
 12  comments                        24 non

In [9]:
df_insights.columns.drop(['ig_reels_avg_watch_time','ig_reels_video_view_total_time'])

Index(['media_id', 'media_type', 'is_reel', 'permalink', 'timestamp',
       'caption', 'status', 'metrics_used', 'metrics_dropped', 'views',
       'reach', 'likes', 'comments', 'saved', 'shares', 'total_interactions',
       'profile_activity', 'profile_visits'],
      dtype='object')

In [10]:
df_insights['timestamp'] = pd.to_datetime(df_insights['timestamp'])

In [11]:
df_insights.dtypes

media_id                                       object
media_type                                     object
is_reel                                          bool
permalink                                      object
timestamp                         datetime64[ns, UTC]
caption                                        object
status                                          int64
metrics_used                                   object
metrics_dropped                                object
views                                           int64
reach                                           int64
likes                                           int64
comments                                        int64
saved                                           int64
shares                                          int64
total_interactions                              int64
ig_reels_avg_watch_time                       float64
ig_reels_video_view_total_time                float64
profile_activity            

In [12]:
import pandas as pd
import numpy as np
import csv
import re

def export_to_tableau(
    df: pd.DataFrame,
    path: str,
    sep: str = ";",
    encoding: str = "utf-8"
):
    """
    Limpia y exporta un DataFrame a CSV de forma segura para Tableau.

    - Elimina saltos de línea en textos (evita filas rotas)
    - Normaliza nombres de columnas
    - Controla nulos (NaN / NaT)
    - Fuerza quoting completo
    - Usa separador europeo (;)

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame limpio en pandas
    path : str
        Ruta de salida del CSV
    sep : str, optional
        Separador del CSV (default ';')
    encoding : str, optional
        Encoding del archivo (default 'utf-8')
    """

    df = df.copy()

    # 1️⃣ Normalizar nombres de columnas
    df.columns = (
        df.columns
          .str.strip()
          .str.lower()
          .str.replace(" ", "_")
          .str.replace(r"[^a-z0-9_]", "", regex=True)
    )

    # 2️⃣ Limpiar saltos de línea y retornos en columnas de texto
    cols_texto = df.select_dtypes(include="object").columns
    df[cols_texto] = df[cols_texto].replace(
        {r"[\r\n]+": " "},
        regex=True
    )

    # 3️⃣ Convertir NaN / NaT a None (Tableau-friendly)
    df = df.replace({pd.NA: None, np.nan: None})

    # 4️⃣ Exportar CSV robusto
    df.to_csv(
        path,
        sep=sep,
        index=False,
        encoding=encoding,
        quoting=csv.QUOTE_ALL
    )

    print(f"✅ CSV exportado correctamente para Tableau: {path}")
    print(f"📊 Filas: {df.shape[0]} | Columnas: {df.shape[1]}")

In [14]:
df_insights.dtypes

media_id                                       object
media_type                                     object
is_reel                                          bool
permalink                                      object
timestamp                         datetime64[ns, UTC]
caption                                        object
status                                          int64
metrics_used                                   object
metrics_dropped                                object
views                                           int64
reach                                           int64
likes                                           int64
comments                                        int64
saved                                           int64
shares                                          int64
total_interactions                              int64
ig_reels_avg_watch_time                       float64
ig_reels_video_view_total_time                float64
profile_activity            

In [13]:
export_to_tableau(df_insights, 'media_insights.csv')

✅ CSV exportado correctamente para Tableau: media_insights.csv
📊 Filas: 24 | Columnas: 20
